In [4]:
import sys
import os
import mysql.connector
import pandas as pd
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')
import json
import pickle
import numpy as np
import datetime
from IPython.display import display, HTML

In [5]:
# Connect ke database config
config = get_db_config()
# Ambil host dari salah satu config (misal db_old)
print(f'Database config loaded: {config["db_old"]["host"]}')

# Connect ke DB Lama
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
print(f'Connected to old database: {config["db_old"]["database"]}')

# Connect ke DB Baru
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f'Connected to new database: {config["db_new"]["database"]}')

db_future = mysql.connector.connect(**config['db_future'])
cursor_future = db_future.cursor(dictionary=True)
print(f'Connected to future database: {config["db_future"]["database"]}')

Database config loaded: localhost
Connected to old database: dataleap_v6_example
Connected to new database: 1
Connected to future database: 1


In [16]:
import pandas as pd

print("🛠️ KLINIK DATA FASE 4: SAPU BERSIH & ROLES 🛠️")

# 1. CLEANUP DATA (Gunakan tanda kutip untuk string!)
ids_to_delete = ['U00001', 'U00027', 'U00028']
idk_to_delete = ['1', '13', '14']

try:
    # SQL IN clause butuh format: ('U00001', 'U00027', ...)
    ids_sql = ", ".join([f"'{i}'" for i in ids_to_delete])
    idk_sql = ", ".join([f"'{i}'" for i in idk_to_delete])
    
    cursor_future.execute(f"DELETE FROM users WHERE id_user IN ({ids_sql})")
    cursor_future.execute(f"DELETE FROM karyawan WHERE id_karyawan IN ({idk_sql})")
    cursor_future.execute("UPDATE karyawan SET nama_lengkap = 'Super Admin', nama_panggilan = 'Admin' WHERE id_karyawan = 15")
    
    db_future.commit()
    print("✅ Data lama berhasil dibersihkan.")
except Exception as e:
    print(f"❌ Error saat cleanup: {e}")

# Definisi perubahan: ID divisi dan Nama baru yang diinginkan
daftar_perubahan = {
    1: 'Global',
    2: 'IT',
    3: 'PENDIDIKAN',
    4: 'HR',
    5: 'FINANCE',
    6: 'BUSDEV',
    7: 'GA'
}

try:
    print("🔄 Memperbarui tabel 'divisions'...")
    
    # Loop untuk melakukan update satu per satu berdasarkan ID
    for id_div, nama_baru in daftar_perubahan.items():
        query = "UPDATE divisions SET name_division = %s WHERE id_division = %s"
        cursor_future.execute(query, (nama_baru, id_div))
        
    db_future.commit() # Simpan perubahan ke database
    print("✅ Berhasil mengubah nama untuk 7 divisi!")

except Exception as e:
    print(f"❌ Terjadi kesalahan saat update divisi: {e}")



import datetime

print("🚀 Sedang memasukkan data ke tabel 'roles'...")

# Waktu saat ini untuk created_at dan updated_at
now = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')

# Daftar data role
roles_data = [
    (1, None, 'super-admin', 'web', now, now),
    (2, None, 'pimpinan', 'web', now, now),
    (3, None, 'division-manager', 'web', now, now),
    (4, None, 'pengajar', 'web', now, now),
    (5, None, 'division-staff', 'web', now, now),
    (6, None, 'division-admin', 'web', now, now),
    (7, None, 'employee', 'web', now, now)
]

insert_query = """
INSERT INTO roles (id, id_division, name, guard_name, created_at, updated_at) 
VALUES (%s, %s, %s, %s, %s, %s)
"""

try:
    cursor_future.executemany(insert_query, roles_data)
    db_future.commit()
    print(f"✅ Berhasil memasukkan {len(roles_data)} role ke tabel 'roles'!")
except Exception as e:
    print(f"❌ Error saat insert roles: {e}")

# 2. PROSES MIGRASI MODEL_HAS_ROLES
try:
    df_roles = pd.read_csv('model_has_roles.csv')
    
    # CEK DATA: Pastikan role_id di CSV ada di tabel 'roles'
    cursor_future.execute("SELECT id FROM roles")
    existing_roles = [r['id'] for r in cursor_future.fetchall()]
    
    # Filter hanya role yang valid
    df_valid = df_roles[df_roles['role_id'].isin(existing_roles)].copy()
    
    if len(df_valid) < len(df_roles):
        print(f"⚠️ Warning: {len(df_roles) - len(df_valid)} baris di-skip karena role_id tidak terdaftar di tabel 'roles'.")

    insert_query = """
    INSERT INTO model_has_roles (role_id, model_type, model_id, id_division) 
    VALUES (%s, %s, %s, %s)
    """
    roles_data = [tuple(x) for x in df_valid[['role_id', 'model_type', 'model_id', 'id_division']].to_numpy()]
    
    cursor_future.executemany(insert_query, roles_data)
    db_future.commit()
    print(f"🚀 Berhasil memasukkan {len(roles_data)} data ke tabel model_has_roles!")
    
except Exception as e:
    print(f"❌ Error saat migrasi roles: {e}")

🛠️ KLINIK DATA FASE 4: SAPU BERSIH & ROLES 🛠️
✅ Data lama berhasil dibersihkan.
🔄 Memperbarui tabel 'divisions'...
✅ Berhasil mengubah nama untuk 7 divisi!
🚀 Sedang memasukkan data ke tabel 'roles'...
✅ Berhasil memasukkan 7 role ke tabel 'roles'!
🚀 Berhasil memasukkan 105 data ke tabel model_has_roles!
